## Setup
Credenciales y URIs

In [101]:
import sys

!{sys.executable} -m pip install "sagemaker>=2.99.0,<3.0"

import boto3
import sagemaker
from sagemaker.workflow.pipeline_context import PipelineSession

sagemaker_session = sagemaker.session.Session()
region = sagemaker_session.boto_region_name
role = sagemaker.get_execution_role()
pipeline_session = PipelineSession()
default_bucket = sagemaker_session.default_bucket()
default_bucket_prefix = sagemaker_session.default_bucket_prefix
default_bucket_prefix_path = ""

if default_bucket_prefix:
    default_bucket_prefix_path = f"/{default_bucket_prefix}"

model_package_group_name = f"AbaloneModelPackageGroupDev"

# Imágenes en ECR
account_id = boto3.client("sts").get_caller_identity()["Account"]
repo_processing = "sagemaker-sklearn-preprocess"
repo_training = "sagemaker-xgboost-byoc"

# URIs dinámicas
processing_image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{repo_processing}:latest"
training_image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{repo_training}:latest"

print(f"Bucket por defecto: {default_bucket}")
print(f"Processing Image: {processing_image_uri}")
print(f"Training Image: {training_image_uri}")

Bucket por defecto: sagemaker-us-east-1-150215480648
Processing Image: 150215480648.dkr.ecr.us-east-1.amazonaws.com/sagemaker-sklearn-preprocess:latest
Training Image: 150215480648.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost-byoc:latest


## Carga de datos y parámetros

In [102]:
from sagemaker.workflow.parameters import (
    ParameterInteger,
    ParameterString,
    ParameterFloat,
)
# Bucket con los datos cargados en la tarea 6
bucket = default_bucket
input_prefix = "sagemaker/processing-data/input/raw"
s3_data_uri = f"s3://{bucket}/{input_prefix}"

print(f"El pipeline usará los datos contenidos en : {s3_data_uri}")

# Verificación de archivos en bucket
response = sagemaker_session.boto_session.client("s3").list_objects_v2(
    Bucket=bucket, 
    Prefix=input_prefix
)
print("\nArchivos detectados en S3:")
if "Contents" in response:
    for obj in response["Contents"]:
        print(f" {obj['Key']}")

# Parámetros
processing_instance_count = ParameterInteger(name="ProcessingInstanceCount", default_value=1)
instance_type = ParameterString(name="TrainingInstanceType", default_value="ml.m5.xlarge")
model_approval_status = ParameterString(name="ModelApprovalStatus", default_value="PendingManualApproval")


input_data = ParameterString(name="InputData", default_value=s3_data_uri)
batch_data = ParameterString(name="BatchData", default_value=s3_data_uri)
rmse_threshold = ParameterFloat(name="RmseThreshold", default_value=5.0)

El pipeline usará los datos contenidos en : s3://sagemaker-us-east-1-150215480648/sagemaker/processing-data/input/raw

Archivos detectados en S3:
 sagemaker/processing-data/input/raw/item_categories.csv
 sagemaker/processing-data/input/raw/item_categories_en.csv
 sagemaker/processing-data/input/raw/items.csv
 sagemaker/processing-data/input/raw/items_en.csv
 sagemaker/processing-data/input/raw/sales_train.csv
 sagemaker/processing-data/input/raw/shops.csv
 sagemaker/processing-data/input/raw/shops_en.csv
 sagemaker/processing-data/input/raw/submission.csv
 sagemaker/processing-data/input/raw/test.csv


## PrepocessingStep

In [103]:
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.workflow.steps import ProcessingStep


script_processor = ScriptProcessor(
    image_uri=processing_image_uri,
    command=["python3"],
    instance_type="ml.m5.xlarge",
    instance_count=processing_instance_count,
    base_job_name="byoc-preprocess",
    role=role,
    sagemaker_session=pipeline_session,
)


processor_args = script_processor.run(
    inputs=[
        ProcessingInput(
            source=input_data, 
            destination="/opt/ml/processing/input"
        ),
    ],
    outputs=[
        ProcessingOutput(output_name="train", source="/opt/ml/processing/output/train"),
        ProcessingOutput(output_name="validation", source="/opt/ml/processing/output/validation"),
        ProcessingOutput(output_name="test", source="/opt/ml/processing/output/test"),
        ProcessingOutput(output_name="test_csv", source="/opt/ml/processing/output/test_csv"),
    ],
    code="../processing/container/preprocess.py",
)


step_process = ProcessingStep(name="PreprocesamientoBYOC", step_args=processor_args)
print("Preprocessing step listo")

Preprocessing step listo


## TrainingStep

In [104]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.workflow.steps import TrainingStep

model_path = f"s3://{default_bucket}/{default_bucket_prefix}/modelos"

xgb_train = Estimator(
    image_uri=training_image_uri,
    instance_type=instance_type,
    instance_count=1,
    output_path=model_path,
    role=role,
    sagemaker_session=pipeline_session,
)


train_args = xgb_train.fit(
    inputs={
        "train": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="application/x-parquet",
        ),
        "validation": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            content_type="application/x-parquet",
        ),
    }
)


step_train = TrainingStep(
    name="EntrenamientoBYOC",
    step_args=train_args,
)
print("TrainingStep listo")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


TrainingStep listo


## EvaluationStep

In [105]:
from sagemaker.workflow.properties import PropertyFile

eval_args = script_processor.run(
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[
        ProcessingOutput(output_name="evaluation", source="/opt/ml/processing/evaluation"),
    ],
    code="../processing/container/evaluate.py",
)


evaluation_report = PropertyFile(
    name="EvaluationReport", output_name="evaluation", path="evaluation.json"
)


step_eval = ProcessingStep(
    name="EvaluacionBYOC",
    step_args=eval_args,
    property_files=[evaluation_report],
)
print("EvaluationStep listo")

EvaluationStep listo


## Model Step

In [106]:
from sagemaker.model import Model
from sagemaker.workflow.model_step import ModelStep

model = Model(
    image_uri=training_image_uri,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    sagemaker_session=pipeline_session,
    role=role,
)

# Step modelo
step_create_model = ModelStep(
    name="CrearModeloBYOC",
    step_args=model.create(instance_type="ml.m5.large"),
)
print("ModelStep listo")

ModelStep listo


## Transform Step

### Definir un Transform Step para realizar Batch Transformation

In [107]:
from sagemaker.transformer import Transformer
from sagemaker.inputs import TransformInput
from sagemaker.workflow.steps import TransformStep

transformer = Transformer(
    model_name=step_create_model.properties.ModelName,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    output_path=f"s3://{default_bucket}/pipeline-byoc/transform_output",
    strategy="MultiRecord",
    max_payload=5,
    assemble_with="Line",
    accept="text/csv"
)

step_transform = TransformStep(
    name="TransformacionBatchBYOC", 
    transformer=transformer, 
    inputs=TransformInput(
        data=step_process.properties.ProcessingOutputConfig.Outputs["test_csv"].S3Output.S3Uri,
        content_type="text/csv",
        split_type="Line"
    )
)
print("TransformStep listo")

TransformStep listo


## Register Model Step

In [108]:
from sagemaker.model_metrics import MetricsSource, ModelMetrics

# Vinculación con JSON de evaluación
model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri="{}/evaluation.json".format(
            step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
        ),
        content_type="application/json",
    )
)

register_args = model.register(
    content_types=["application/x-parquet"],
    response_types=["application/json"],
    inference_instances=["ml.t2.medium", "ml.m5.large"],
    transform_instances=["ml.m5.xlarge"],
    model_package_group_name=model_package_group_name,
    approval_status=model_approval_status,
    model_metrics=model_metrics,
)

step_register = ModelStep(name="RegistroModeloBYOC", step_args=register_args)
print("ModelStep (Register) listo")

ModelStep (Register) listo


## Fail Step

In [109]:
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.functions import Join

step_fail = FailStep(
    name="FalloPorRMSE",
    error_message=Join(on=" ", values=["Execution failed due to RMSE >", rmse_threshold]),
)


## Condition Step

In [110]:
from sagemaker.workflow.conditions import ConditionLessThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet

cond_lte = ConditionLessThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="regression_metrics.rmse.value",
    ),
    right=rmse_threshold,
)

# unión de steps
step_cond = ConditionStep(
    name="CondicionRMSE",
    conditions=[cond_lte],
    if_steps=[step_register, step_create_model, step_transform], # éxito
    else_steps=[step_fail], # fallo
)


## Definir un Pipeline de parámetros, steps y condiciones

In [116]:
from sagemaker.workflow.pipeline import Pipeline

pipeline_name = f"PipelinePrediccionVentasBYOC"
pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        processing_instance_count,
        instance_type,
        model_approval_status,
        input_data,
        batch_data,
        rmse_threshold,
    ],
    steps=[step_process, step_train, step_eval, step_cond],
)


In [112]:
import json


definition = json.loads(pipeline.definition())
definition

{'Version': '2020-12-01',
 'Metadata': {},
 'Parameters': [{'Name': 'ProcessingInstanceCount',
   'Type': 'Integer',
   'DefaultValue': 1},
  {'Name': 'TrainingInstanceType',
   'Type': 'String',
   'DefaultValue': 'ml.m5.xlarge'},
  {'Name': 'ModelApprovalStatus',
   'Type': 'String',
   'DefaultValue': 'PendingManualApproval'},
  {'Name': 'InputData',
   'Type': 'String',
   'DefaultValue': 's3://sagemaker-us-east-1-150215480648/sagemaker/processing-data/input/raw'},
  {'Name': 'BatchData',
   'Type': 'String',
   'DefaultValue': 's3://sagemaker-us-east-1-150215480648/sagemaker/processing-data/input/raw'},
  {'Name': 'RmseThreshold', 'Type': 'Float', 'DefaultValue': 5.0}],
 'PipelineExperimentConfig': {'ExperimentName': {'Get': 'Execution.PipelineName'},
  'TrialName': {'Get': 'Execution.PipelineExecutionId'}},
 'Steps': [{'Name': 'PreprocesamientoBYOC',
   'Type': 'Processing',
   'Arguments': {'ProcessingResources': {'ClusterConfig': {'InstanceType': 'ml.m5.xlarge',
      'Instance

In [113]:
pipeline.upsert(role_arn=role)

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:150215480648:pipeline/PipelinePrediccionVentasBYOC',
 'ResponseMetadata': {'RequestId': '9b764594-c292-46a5-a9a2-06468220a607',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '9b764594-c292-46a5-a9a2-06468220a607',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '118',
   'date': 'Sun, 22 Mar 2026 01:22:11 GMT'},
  'RetryAttempts': 0}}

In [114]:
execution = pipeline.start()

In [115]:
execution.describe()

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:150215480648:pipeline/PipelinePrediccionVentasBYOC',
 'PipelineExecutionArn': 'arn:aws:sagemaker:us-east-1:150215480648:pipeline/PipelinePrediccionVentasBYOC/execution/xna0cnow2lxq',
 'PipelineExecutionDisplayName': 'execution-1774142532481',
 'PipelineExecutionStatus': 'Executing',
 'PipelineExperimentConfig': {'ExperimentName': 'PipelinePrediccionVentasBYOC',
  'TrialName': 'xna0cnow2lxq'},
 'CreationTime': datetime.datetime(2026, 3, 22, 1, 22, 12, 414000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2026, 3, 22, 1, 22, 12, 414000, tzinfo=tzlocal()),
 'CreatedBy': {'UserProfileArn': 'arn:aws:sagemaker:us-east-1:150215480648:user-profile/d-zopnc380my02/datascientist',
  'UserProfileName': 'datascientist',
  'DomainId': 'd-zopnc380my02',
  'IamIdentity': {'Arn': 'arn:aws:sts::150215480648:assumed-role/SageMakerStudioExecutionRole2026/SageMaker',
   'PrincipalId': 'AROASF6MKKVEO4APIVADN:SageMaker'}},
 'LastModifiedBy': {'UserProf